# 11 (Advanced) Hands-on time-series foundation model: zero-shot forecasting with TimesFM 3.0

In Ch11's [`11_dl_sequence.ipynb`](11_dl_sequence.ipynb) we **trained** an LSTM / CNN ourselves before they beat the persistence baseline. This notebook asks a different question:

> If a model has **already been pre-trained on hundreds of billions of time points**, and we write no training code at all, does it forecast better?

That is **zero-shot forecasting** with a **time-series foundation model (TSFM)**. We run **TimesFM 3.0**, released by Google in August 2026, for real and **evaluate it honestly**:

**Workflow: what zero-shot means → load the model → prediction intervals → rolling-origin backtest on the same test period as Ch11 → add a temperature covariate → a second series from Ch07 → when not to trust it → licence (important!)**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# Extra Colab step: install TimesFM 3.0 (locally with uv: uv sync --group timesfm)
import sys
if 'google.colab' in sys.modules:
    !pip install -q "timesfm[torch]>=3.0.2"

## What does zero-shot mean?

The LSTM in Ch11 Part A must `fit` first: it rehearses the first 300 days of this series for a few hundred epochs before it learns that "temperature 7 days ago pushes cases up".

A TSFM works the other way round: Google pre-trained the model on large, cross-domain collections of time series (Wikipedia page views, Google Trends, public forecasting datasets, synthetic data…), and **what you download is the already-trained weights**. To forecast you just pass in the history and the model returns the future, with **no `fit()` step**:

| | Train your own (Ch11 LSTM) | Zero-shot TSFM (TimesFM 3.0) |
|---|---|---|
| Training needed | Yes, for every series | No |
| History needed | Hundreds of windows to be stable | Works with a few dozen points (more is usually better) |
| Output | One point forecast | Median + 9 quantiles (0.1–0.9) → **prediction intervals** |
| Knows this series' mechanism? | Can learn it (e.g. the 7-day lag) | Can only "guess" from the shape of the history |
| Hardware | Seconds on a laptop | ~1.3 GB weight download, ~3 GB of RAM |

> 🧭 **Two execution paths in this notebook**: on Colab, or on a computer with `timesfm` installed, every cell below **really calls TimesFM 3.0**. When this website is built, the 1.3 GB model is not downloaded; instead `epi_learning.tsfm` replays the forecasts saved from a real run (`data/tsfm_cache/ch11_timesfm.json`). Every cell prints which path it took, and the numbers are the same as the real run (rounded to 3 decimals).

In [ ]:
# --- Step 1: imports, and rebuild Ch11's synthetic 'dengue x temperature' series ---
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from epi_learning import tsfm

# -- Font setup (CJK fonts first, so mixed-language labels never render as boxes) --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# -- Exactly the same data generating process as 11_dl_sequence.ipynb (same seed) --
np.random.seed(1)
n = 360
t = np.arange(n)
temp = 24 + 7 * np.sin(2 * np.pi * (t - 30) / 365) + np.random.normal(0, 1.0, n)
drive = np.clip(temp - 24, 0, None)
cases = np.zeros(n)
for i in range(n):
    lag = cases[i - 1] if i >= 1 else 0
    cases[i] = max(
        0,
        0.55 * lag
        + 3.2 * (drive[i - 7] if i >= 7 else 0)
        + 4 * np.sin(2 * np.pi * t[i] / 7)
        + 6
        + np.random.normal(0, 2.0),
    )

H = 7            # as in Ch11: forecast 7 days ahead
split = n - 60   # as in Ch11: the last 60 days are the test period
print(f"Series length: {n} days, test period: day {split}-{n - 1}")
print(f"Cases: mean={cases.mean():.1f}, min={cases.min():.1f}, max={cases.max():.1f}")

## Step 2 — Load TimesFM 3.0

The official API is three lines: `ModelConfig` names the weights (`google/timesfm-3.0-pytorch` on Hugging Face), the batch size and the device, and `TimesFM3Evaluator` loads the model. The first run downloads ~1.3 GB of weights into `~/.cache/huggingface`; later runs read them from disk.

`TimesFM3Evaluator` uses the official benchmark defaults: it always returns quantiles, applies "symmetric averaging" (forecast the series and its negation, then average, which is more stable), and keeps forecasts non-negative when the whole history is non-negative, which suits case counts.

In [ ]:
# --- Step 2: load TimesFM 3.0 (replay the cache when timesfm is not installed) ---
CACHE = tsfm.cache_path("ch11_timesfm")   # data/tsfm_cache/ch11_timesfm.json
MODE = tsfm.resolve_mode()                # "model": timesfm installed; "cache": not installed (e.g. the website build)

if MODE == "cache":
    model = None
    print("Path = CACHE: timesfm3 is not run this time; the forecasts below come from a real TimesFM 3.0 run")
    print(tsfm.describe_cache(CACHE))
else:
    import time
    from timesfm3 import TimesFM3Evaluator, ModelConfig

    t0 = time.perf_counter()
    config = ModelConfig(
        checkpoint_path="google/timesfm-3.0-pytorch",  # weights on Hugging Face (~1.3 GB)
        per_core_batch_size=32,                        # how many series one forward pass handles
        device=tsfm.default_device(),                  # "cuda" if a GPU is available, else "cpu"
    )
    model = TimesFM3Evaluator(config)
    n_params = sum(p.numel() for p in model.model.parameters())
    print(f"Path = MODEL: TimesFM 3.0 loaded, {n_params / 1e6:.0f}M parameters, "
          f"device {model.device}, took {time.perf_counter() - t0:.0f} s")

## Step 3 — First zero-shot forecast: give it 300 days, ask for the next 60

`model.predict_batch()` takes a list of series (each may have a different length) and returns, per series:

- `.forecast`: the median forecast, shape `(horizon,)`
- `.quantiles`: 9 quantiles (0.1, 0.2, …, 0.9), shape `(horizon, 9)`; columns 0 and 8 bound the **80% prediction interval**

Below we first call `model.predict_batch()` directly, then do the same through `tsfm.forecast_batch()` (with a model it simply calls `predict_batch`; without one it reads the cache).

In [ ]:
# --- Step 3: zero-shot forecast -- first 300 days -> next 60 days ---
context = cases[:split]   # the history the model sees: day 0-299 (no training at all)
H_LONG = 60

if model is not None:
    # The genuine TimesFM 3.0 call: no fit(), just predict
    out = next(model.predict_batch([context.astype(np.float32)], horizon=H_LONG, return_quantiles=True))
    print("forecast shape:", out.forecast.shape, "| quantiles shape:", out.quantiles.shape)

first = tsfm.forecast_batch(model, [context], H_LONG, cache_key="first_look", cache_file=CACHE)
median = first.forecast[0]
lo, hi = first.quantile(0.1)[0], first.quantile(0.9)[0]
days = np.arange(split, split + H_LONG)

print(f"Quantile levels: {tsfm.QUANTILE_LEVELS}")
print(f"MAE of the median forecast over the next 60 days: {np.mean(np.abs(median - cases[days])):.2f}")
print(f"Share of true values inside the 80% prediction interval (0.1-0.9 quantiles): "
      f"{tsfm.coverage(cases[days], lo, hi):.0%} (ideal: about 80%)")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(np.arange(200, split), cases[200:split], color="#6B6B6B", label="History the model sees")
ax.plot(days, cases[days], color="#1A1A1A", linewidth=2, label="True cases (never seen)")
ax.plot(days, median, color="#D97757", linewidth=2, label="TimesFM median forecast")
ax.fill_between(days, lo, hi, color="#D97757", alpha=0.2, label="80% prediction interval")
ax.axvline(split, color="#6A9BCC", linestyle="--")
ax.set_xlabel("Day index")
ax.set_ylabel("Daily cases")
ax.set_title("TimesFM 3.0 zero-shot: no training, 60 days ahead")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Step 4 — Honest evaluation: the same test period and metrics as Ch11

A nice-looking plot proves nothing. To compare fairly with Ch11's LSTM / CNN, the setup must be identical:

- **Same series, same test period**: days 300–359, 60 target days
- **Same task**: from the history up to the origin, forecast the case count **7 days ahead**
- **Rolling-origin backtest**: every target day has its own origin, and the model only sees data before that origin, the same principle as Ch11's "time-series split" section: the future must never leak in
- **Same metrics**: MAE, RMSE, MAPE; TimesFM also gives prediction intervals, so we add **coverage** and the **weighted quantile loss (WQL, an approximation of CRPS)**

The LSTM / CNN numbers are quoted from what [`11_dl_sequence.ipynb`](11_dl_sequence.ipynb) prints for the same test period; persistence is recomputed here to confirm that the test period really is identical (it should be 2.879).

In [ ]:
# --- Step 4: rolling-origin backtest (60 origins, 7 days ahead each) ---
origins = np.arange(split - H + 1, n - H + 1)              # origins day 294-353
targets = origins + H - 1                                   # day 7 = day 300-359 (= Ch11's idxte)
contexts = tsfm.rolling_origin_contexts(cases, origins)     # [cases[:294], cases[:295], ...]

bt = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_univariate", cache_file=CACHE)
tfm_pred = bt.forecast[:, H - 1]                            # the "day 7" median forecast of each origin
tfm_lo, tfm_hi = bt.quantile(0.1)[:, H - 1], bt.quantile(0.9)[:, H - 1]

persistence_pred = cases[targets - H]   # Ch11's naive baseline (7 days ahead = last known value; = seasonal naive, period 7)


def mae(pred, idx=targets):
    return np.mean(np.abs(pred - cases[idx]))


def rmse(pred, idx=targets):
    return np.sqrt(np.mean((pred - cases[idx]) ** 2))


def mape(pred, idx=targets):
    return np.mean(np.abs(pred - cases[idx]) / np.maximum(cases[idx], 1e-6)) * 100


# Scores printed by 11_dl_sequence.ipynb for the same test period (MAE, RMSE, MAPE%)
CH11_REFERENCE = {
    "LSTM (Ch11, trained)": (2.149, 2.727, 21.45),
    "1D-CNN (Ch11, trained)": (2.197, 2.639, 20.42),
}

rows = [
    ("Persistence (naive baseline)", mae(persistence_pred), rmse(persistence_pred), mape(persistence_pred)),
    ("TimesFM 3.0 (zero-shot)", mae(tfm_pred), rmse(tfm_pred), mape(tfm_pred)),
] + [(name, *values) for name, values in CH11_REFERENCE.items()]
table = pd.DataFrame(rows, columns=["Model", "MAE", "RMSE", "MAPE(%)"]).round(3)
print(table.to_string(index=False))

print(f"\nTimesFM 80% prediction-interval coverage: {tsfm.coverage(cases[targets], tfm_lo, tfm_hi):.0%} (ideal: about 80%)")
print(f"TimesFM WQL (lower is better): {tsfm.mean_weighted_quantile_loss(cases[targets], bt.quantiles[:, H - 1]):.3f}")
print(f"TimesFM beats persistence? {mae(tfm_pred) < mae(persistence_pred)}")
print(f"TimesFM beats Ch11's LSTM? {mae(tfm_pred) < CH11_REFERENCE['LSTM (Ch11, trained)'][0]}")

In [ ]:
# --- Step 4b: the 7-day-ahead forecasts over the test period ---
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(targets, cases[targets], color="#1A1A1A", linewidth=2, label="True cases")
ax.plot(targets, persistence_pred, color="#6B6B6B", linestyle="--", label="Persistence (naive)")
ax.plot(targets, tfm_pred, color="#D97757", linewidth=2, label="TimesFM 3.0 (zero-shot)")
ax.fill_between(targets, tfm_lo, tfm_hi, color="#D97757", alpha=0.2, label="TimesFM 80% prediction interval")
ax.set_xlabel("Day index")
ax.set_ylabel("Daily cases")
ax.set_title("Rolling-origin backtest: every point was forecast 7 days earlier")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### How to read this scorecard

- **It beats persistence**: without a single line of training code, TimesFM has a lower MAE than the naive baseline, and it comes with prediction intervals whose coverage is close to 80%. That is the main value of a zero-shot model: **a strong baseline that needs no tuning and is ready immediately**
- **It loses to Ch11's LSTM / CNN**: those models were **trained on this very series** and learned the mechanism "temperature 7 days ago pushes cases up"; TimesFM only sees the shape of the case history and does not know that mechanism
- This is the spirit of Ch11's decision framework: with enough history and a clear leading indicator, a model trained for the task can win; zero-shot models shine **when history is short, there is no time to train, or you must forecast hundreds of series at once**

### Can we tune the context length? Decide on the validation period, never on the test period

By default TimesFM reads the whole history. Would only the last few weeks be more accurate? That is a **hyperparameter**, like the number of epochs in Ch11, so it **must be chosen on the validation period** (Ch11's validation period: target days 260–299), and only then do we touch the test period once.

In [ ]:
# --- Step 5: choose the context length on the validation period, then check the test period once ---
val_origins = np.arange(split - 40 - H + 1, split - H + 1)   # target days 260-299 (all before the test period)
val_targets = val_origins + H - 1
candidates = [28, 56, 112, None]                              # None = the whole history

val_scores = {}
for max_context in candidates:
    ctx = tsfm.rolling_origin_contexts(cases, val_origins, max_context=max_context)
    res = tsfm.forecast_batch(model, ctx, H, cache_key=f"val_context_{max_context or 'all'}",
                              cache_file=CACHE, verbose=False)
    val_scores[max_context] = mae(res.forecast[:, H - 1], val_targets)
    print(f"context = {str(max_context or 'all'):>4} days -> validation MAE = {val_scores[max_context]:.3f}")

best = min(val_scores, key=val_scores.get)
test_ctx = tsfm.rolling_origin_contexts(cases, origins, max_context=best)
tuned = tsfm.forecast_batch(model, test_ctx, H, cache_key=f"test_context_{best or 'all'}", cache_file=CACHE)
print(f"\nValidation picked context = {best or 'all'} -> test MAE = {mae(tuned.forecast[:, H - 1]):.3f}"
      f" (default, whole history: {mae(tfm_pred):.3f})")

## Step 6 — Covariates: tell it the temperature too

New in TimesFM 3.0 is **native multivariate and covariate support**, with no per-task tuning:

- `past_only_covariates`: variables known only up to the origin (e.g. past temperature), shape `(n_covariates, T)`
- `past_future_covariates`: variables also known over the forecast horizon (e.g. a weather forecast, holidays, scheduled vaccination days), shape `(n_covariates, T + horizon)`
- You can also pass several **target** series at once (shape `(n_targets, T)`, e.g. cases per district); the model forecasts them jointly, letting them inform each other

In this dataset cases are driven by "temperature 7 days earlier", and when we forecast 7 days ahead, "7 days earlier" is exactly the origin, so **past** temperature alone should in principle be enough. We try both: past temperature only, and a hypothetical **perfect 7-day temperature forecast**.

In [ ]:
# --- Step 6: add the temperature covariate (same 60 origins) ---
past_only = [temp[None, :o] for o in origins]          # per origin: past temperature, shape (1, T)
past_future = [temp[None, :o + H] for o in origins]    # per origin: past + next 7 days of temperature, shape (1, T + 7)

# With a model, tsfm.forecast_batch simply calls:
#   model.predict_batch(contexts, horizon=7, past_only_covariates=past_only, return_quantiles=True)
cov_po = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_temp_past_only",
                             cache_file=CACHE, past_only_covariates=past_only)
cov_pf = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_temp_past_future",
                             cache_file=CACHE, past_future_covariates=past_future)

rows = [
    ("TimesFM: cases only", bt),
    ("TimesFM: + past temp", cov_po),
    ("TimesFM: + perfect temp forecast", cov_pf),
]
for name, res in rows:
    pred = res.forecast[:, H - 1]
    cov80 = tsfm.coverage(cases[targets], res.quantile(0.1)[:, H - 1], res.quantile(0.9)[:, H - 1])
    print(f"{name:<34} MAE = {mae(pred):.3f}   80% coverage = {cov80:.0%}   (model time {res.seconds:.1f} s)")
print(f"{'LSTM (Ch11, with temp, trained)':<34} MAE = {CH11_REFERENCE['LSTM (Ch11, trained)'][0]:.3f}")

### The covariate did not help — why?

On this dataset, TimesFM with temperature is **not more accurate than with cases alone** (it is slightly worse). The honest reading:

1. A zero-shot model does not know the **dataset-specific** mechanism "temperature only matters above 24 °C, with a lag of exactly 7 days"; it can only guess the relation between the two series from the same stretch of history, and here the temperature signal is buried under noise and the weekly rhythm
2. Ch11's LSTM was **trained on this series**, which is how it learned that mechanism
3. Practical lesson: covariates do not help automatically. **Prove every covariate with the same rolling-origin backtest**, and remember that a "future covariate" is itself a forecast in the real world (weather forecasts are wrong too)

## Step 7 — A second series: Ch07's 90-day influenza-like-illness series

Part B of Ch07 synthesised 90 days of daily influenza-like-illness notifications (trend + 7-day cycle + noise) and forecast the last 7 days with ARIMA / SARIMA / Prophet. Same series (same seed), now with TimesFM:

- **(a) Ch07's single split**: first 83 days → last 7 days, compared directly with the MAE values Ch07 prints
- **(b) Rolling origin**: a single week is too easily luck, so we also use 14 origins (day 56, 58, …, 82), each forecasting the next 7 days, against **seasonal naive** ("next week = the same weekday last week") and a **SARIMA** refitted at every origin

In [ ]:
# --- Step 7: Ch07's 90-day ILI series (identical to Ch07's synth_cases) ---
import warnings

from statsmodels.tsa.statespace.sarimax import SARIMAX

rng = np.random.default_rng(42)
n_days = 90
trend = np.linspace(3, 7, n_days)
seasonal = 3 * np.sin(2 * np.pi * np.arange(n_days) / 7)
noise = rng.normal(0, 1.2, n_days)
ili = np.maximum(0, (trend + seasonal + noise).round()).astype(int)

# (a) Ch07's single split: first 83 days -> last 7 days
one = tsfm.forecast_batch(model, [ili[:-7]], 7, cache_key="ili_last_week", cache_file=CACHE)
CH07_REFERENCE = {"ARIMA(1,1,1)": 1.872, "SARIMA(1,1,1)(1,1,1,7)": 0.770, "Prophet": 0.774}  # MAE printed by Ch07
print("(a) MAE over the last 7 days (same split)")
print(f"    TimesFM 3.0 (zero-shot)        {np.mean(np.abs(one.forecast[0] - ili[-7:])):.3f}")
print(f"    Seasonal naive (same weekday)  {np.mean(np.abs(ili[-14:-7] - ili[-7:])):.3f}")
for name, value in CH07_REFERENCE.items():
    print(f"    {name:<31}{value:.3f} (Ch07)")

# (b) Rolling origin: 14 origins, 7 days ahead each
ili_origins = np.arange(56, 84, 2)
ili_actual = np.stack([ili[o:o + 7] for o in ili_origins])
roll = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(ili, ili_origins), 7,
                           cache_key="ili_rolling", cache_file=CACHE)
snaive = np.stack([ili[o - 7:o] for o in ili_origins])
sarima = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # SARIMA often warns about convergence on short series
    for o in ili_origins:
        fit = SARIMAX(ili[:o].astype(float), order=(1, 1, 1), seasonal_order=(1, 1, 1, 7)).fit(disp=False)
        sarima.append(fit.forecast(7))
sarima = np.array(sarima)

print(f"\n(b) MAE over the rolling origins ({len(ili_origins)} origins x 7 days)")
print(f"    TimesFM 3.0 (zero-shot)        {np.mean(np.abs(roll.forecast - ili_actual)):.3f}")
print(f"    SARIMA (refit at every origin) {np.mean(np.abs(sarima - ili_actual)):.3f}")
print(f"    Seasonal naive                 {np.mean(np.abs(snaive - ili_actual)):.3f}")
print(f"    TimesFM 80% interval coverage: {tsfm.coverage(ili_actual, roll.quantile(0.1), roll.quantile(0.9)):.0%}")

ILI_READ_PLACEHOLDER

## Step 8 — When should you **not** trust it?

A TSFM learned "what time series usually look like": smooth, periodic, continuing their trend. Epidemics often refuse to follow that script. Using the book's main dataset, here are the two most dangerous situations:

1. **Outbreak onset**: the nursing home's daily onset counts, forecast 5 days ahead from day 5 (just before the peak) and from day 9 (just after the peak)
2. **Reporting delay**: the most recent cases have not all been notified yet (Ch04's reporting delay), and the model mistakes "not yet reported" for "the epidemic is declining"

In [ ]:
# --- Step 8a: the Legionella outbreak -- a zero-shot model cannot see "about to explode" ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
onset = (
    df[df["clinical_severity"] != "not_ill"]
    .groupby("symptom_onset_date").size()
    .asfreq("D", fill_value=0)
)
y = onset.to_numpy(dtype=float)
print(f"Daily onsets ({len(y)} days): {y.astype(int).tolist()}")

leg_origins = [5, 9]   # day 5: just before the peak; day 9: just after it
leg = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(y, leg_origins), 5,
                          cache_key="legionella_onset", cache_file=CACHE)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, k, o in zip(axes, range(len(leg_origins)), leg_origins):
    future = np.arange(o, o + 5)
    ax.bar(np.arange(len(y)), y, color="#6A9BCC", alpha=0.5, label="Actual daily onsets")
    ax.plot(future, leg.forecast[k], color="#D97757", marker="o", linewidth=2, label="TimesFM median")
    ax.fill_between(future, leg.quantile(0.1)[k], leg.quantile(0.9)[k], color="#D97757", alpha=0.2,
                    label="80% prediction interval")
    ax.axvline(o - 0.5, color="#1A1A1A", linestyle="--")
    ax.set_title(f"Origin: day {o} (sees only the first {o} days)")
    ax.set_xlabel("Outbreak day")
    inside = tsfm.coverage(y[future], leg.quantile(0.1)[k], leg.quantile(0.9)[k])
    print(f"Origin day {o}: forecast {np.round(leg.forecast[k], 1)}, actual {y[future].astype(int)}, "
          f"inside the 80% interval {inside:.0%}")
axes[0].set_ylabel("Daily onsets")
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 8b: reporting delay -- the last 7 days are not fully notified yet ---
origin = 330
completeness = np.array([0.95, 0.9, 0.8, 0.65, 0.5, 0.35, 0.2])   # share already notified (the more recent, the less complete)
reported = cases[:origin].copy()
reported[-7:] = reported[-7:] * completeness                      # what we see "today"

delay = tsfm.forecast_batch(model, [cases[:origin], reported], 7, cache_key="reporting_delay", cache_file=CACHE)
truth = cases[origin:origin + 7]
print(f"True cases over the next 7 days:                {truth.sum():6.1f}")
print(f"Forecast total from complete data:              {delay.forecast[0].sum():6.1f}")
print(f"Forecast total from not-yet-complete data:      {delay.forecast[1].sum():6.1f}"
      f" (under by {1 - delay.forecast[1].sum() / truth.sum():.0%})")
print("-> The fix is not in the model: nowcast first (correct for reporting delay), or only use settled dates as history.")

SUMMARY_PLACEHOLDER